# Lab 7 — Delta Lake

> Extraído de `aulas/Aula_5_etl_medallion_delta.ipynb` (seções 7 e 8, exercício 4 e resumo). As referências a números de seção usam a numeração do notebook original.

## Preparação

Este notebook roda sozinho: as células abaixo recriam o estado necessário, vindo da primeira parte.

In [ ]:
import os, json, csv, random, shutil
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (
    SparkSession.builder
    .appName("Aula 5 - ETL Medallion + Delta")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()   # baixa os jars do Delta na 1a vez
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
LAKE       = "/tmp/lake"
RAW        = f"{LAKE}/raw"
BRONZE     = f"{LAKE}/bronze/orders"
SILVER     = f"{LAKE}/silver/orders"
QUARANTINE = f"{LAKE}/quarantine/orders"
GOLD       = f"{LAKE}/gold"

def tree(path):
    """Mostra a estrutura de pastas com número de arquivos e tamanho."""
    for root, dirs, files in sorted(os.walk(path)):
        dirs.sort()
        data = [f for f in files if not f.startswith((".", "_"))]
        depth = root[len(path):].count(os.sep)
        size = sum(os.path.getsize(os.path.join(root, f)) for f in data) / 1024
        info = f"  ({len(data)} arquivos, {size:,.0f} KB)" if data else ""
        print("    " * depth + (os.path.basename(root) or root) + "/" + info)

def scan_info(df):
    """Mostra os filtros e colunas que chegam até a leitura dos arquivos."""
    import io, contextlib, re
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    scan = next(l for l in buf.getvalue().splitlines() if "FileScan" in l)
    for key in ["PartitionFilters", "PushedFilters", "ReadSchema"]:
        m = re.search(key + r": (\[.*?\]|struct<.*?>)", scan)
        print(f"{key:<17}: {m.group(1) if m else '-'}")

In [ ]:
REGIONS  = ["North", "Northeast", "Midwest", "Southeast", "South"]
STATUSES = ["created", "paid", "shipped", "delivered", "cancelled"]
PRODUCTS = [
    (1, "Notebook", "Computers", 3500), (2, "Monitor", "Computers", 1200),
    (3, "Mouse", "Peripherals", 85),    (4, "Keyboard", "Peripherals", 150),
    (5, "Webcam", "Peripherals", 320),  (6, "USB Hub", "Peripherals", 120),
    (7, "Headset", "Audio", 250),       (8, "Speaker", "Audio", 400),
    (9, "External HD", "Storage", 400), (10, "Pendrive", "Storage", 60),
    (11, "USB Cable", "Cables", 25),    (12, "HDMI Cable", "Cables", 40),
]

rng = random.Random(42)
next_id = [100_000]
known_orders = {}   # order_id -> created_at (para gerar atualizações em dias seguintes)

def dirty_region(r):
    if rng.random() < 0.01: return "N/D"           # região não informada
    return rng.choice([r, r, r, r.lower(), " " + r.upper(), r + "  "])

def dirty_price(p):
    roll = rng.random()
    if roll < 0.03: return "N/A"
    if roll < 0.05: return f"-{p:.2f}"
    if roll < 0.25: return f"R$ {p:.2f}"
    return f"{p:.2f}"

def dirty_quantity():
    roll = rng.random()
    if roll < 0.02: return ""
    if roll < 0.04: return "0"
    return str(rng.randint(1, 4))

def make_order(order_id, created, updated, status, fmt):
    pid, _, _, base = rng.choice(PRODUCTS)
    if rng.random() < 0.02:
        pid = 999                                   # produto inexistente
    ts_created = created.strftime(fmt) if rng.random() > 0.01 else "2024-13-45 99:99"   # data corrompida
    return {
        "order_id":    str(order_id),
        "customer_id": str(rng.randint(1, 3000)) if rng.random() > 0.01 else "",
        "product_id":  str(pid),
        "quantity":    dirty_quantity(),
        "price":       dirty_price(base * rng.uniform(0.9, 1.1)),
        "region":      dirty_region(rng.choice(REGIONS)),
        "status":      status if rng.random() > 0.01 else "pending_review",       # status fora do domínio
        "created_at":  ts_created,
        "updated_at":  updated.strftime(fmt),
    }

def generate_day(day, n_new=1200, n_updates=300):
    """Gera os arquivos de um dia: pedidos novos + atualizações de pedidos antigos."""
    rows_csv, rows_json = [], []
    for _ in range(n_new):
        oid = next_id[0]; next_id[0] += 1
        created = day + timedelta(minutes=rng.randint(0, 1439))
        is_app = rng.random() < 0.6
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else rng.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"])
        row = make_order(oid, created, created, rng.choice(["created", "paid"]), fmt)
        known_orders[oid] = (created, is_app, row)
        (rows_json if is_app else rows_csv).append(row)

    # atualizações de status de pedidos de dias anteriores
    old = [o for o, (c, _, _) in known_orders.items() if c.date() < day.date()]
    for oid in rng.sample(old, min(n_updates, len(old))):
        created, is_app, original = known_orders[oid]
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else "%Y-%m-%d %H:%M:%S"
        row = dict(original)
        row["status"] = rng.choice(["shipped", "delivered", "cancelled"])
        row["created_at"] = created.strftime(fmt)
        row["updated_at"] = (day + timedelta(minutes=rng.randint(0, 1439))).strftime(fmt)
        (rows_json if is_app else rows_csv).append(row)

    # duplicatas exatas (reenvio de arquivo, retry de API...)
    rows_csv  += rng.sample(rows_csv,  len(rows_csv)  // 30)
    rows_json += rng.sample(rows_json, len(rows_json) // 30)

    tag = day.strftime("%Y%m%d")
    with open(f"{RAW}/orders_csv/pdv_{tag}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows_csv[0].keys()))
        w.writeheader(); w.writerows(rows_csv)
    with open(f"{RAW}/orders_json/app_{tag}.json", "w") as f:
        for r in rows_json:
            r = dict(r, device=rng.choice(["ios", "android", "web"]))
            if rng.random() < 0.5 and r["price"].replace(".", "").isdigit():
                r["price"] = float(r["price"])      # no JSON, às vezes número, às vezes texto
            f.write(json.dumps(r) + "\n")
    return len(rows_csv), len(rows_json)

shutil.rmtree(LAKE, ignore_errors=True)
os.makedirs(f"{RAW}/orders_csv"); os.makedirs(f"{RAW}/orders_json")

with open(f"{RAW}/products.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["product_id", "product_name", "category", "list_price"]); w.writerows(PRODUCTS)

for d in ["2024-07-01", "2024-07-02", "2024-07-03"]:
    n_csv, n_json = generate_day(datetime.fromisoformat(d))
    print(f"{d}: {n_csv} linhas CSV, {n_json} linhas JSON")

tree(RAW)

In [ ]:
raw_csv = (
    spark.read
    .option("header", True)                 # sem inferSchema: tudo string
    .csv(f"{RAW}/orders_csv")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("store"))
)

raw_json = (
    spark.read
    .option("primitivesAsString", True)     # números e booleanos também viram string
    .json(f"{RAW}/orders_json")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("app"))
)

bronze = (
    raw_csv.unionByName(raw_json, allowMissingColumns=True)
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze.write.mode("overwrite").parquet(BRONZE)
bronze = spark.read.parquet(BRONZE)

bronze.printSchema()
bronze.groupBy("channel").count().show()

In [ ]:
TS_FORMATS = ["yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm", "yyyy-MM-dd'T'HH:mm:ss"]

def parse_ts(col):
    """Tenta cada formato e fica com o primeiro que funcionar."""
    return F.coalesce(*[F.try_to_timestamp(F.col(col), F.lit(fmt)) for fmt in TS_FORMATS])

def clean(df):
    return (
        df
        .withColumn("order_id",    F.col("order_id").try_cast("long"))
        .withColumn("customer_id", F.col("customer_id").try_cast("long"))
        .withColumn("product_id",  F.col("product_id").try_cast("int"))
        .withColumn("quantity",    F.col("quantity").try_cast("int"))
        .withColumn("price",       F.regexp_replace("price", r"R\$|\s", "").try_cast("double"))
        .withColumn("region",      F.initcap(F.trim("region")))
        .withColumn("status",      F.lower(F.trim("status")))
        .withColumn("created_at",  parse_ts("created_at"))
        .withColumn("updated_at",  parse_ts("updated_at"))
    )

products = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW}/products.csv")

def enrich(df):
    return (
        df.join(F.broadcast(products.select("product_id", "product_name", "category")), "product_id", "left")
        .withColumn("revenue", F.round(F.col("price") * F.col("quantity"), 2))
        .withColumn("order_date", F.to_date("created_at"))
    )

clean(bronze).select("order_id", "price", "quantity", "region", "created_at").show(5)

In [ ]:
RULES = {
    "order_id_nulo":        F.col("order_id").isNull(),
    "preco_invalido":       F.col("price").isNull() | (F.col("price") <= 0),
    "quantidade_invalida":  F.col("quantity").isNull() | (F.col("quantity") <= 0),
    "data_invalida":        F.col("created_at").isNull() | F.col("updated_at").isNull(),
    "regiao_desconhecida":  F.col("region").isNull() | ~F.col("region").isin(REGIONS),
    "produto_desconhecido": F.col("category").isNull(),
}

def validate(df):
    errors = F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in RULES.items()]))
    df = df.withColumn("_errors", errors)
    valid    = df.filter(F.size("_errors") == 0).drop("_errors")
    rejected = df.filter(F.size("_errors") > 0)
    return valid, rejected

def deduplicate(df):
    """Mantém só a versão mais recente de cada pedido."""
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

SILVER_COLS = ["order_id", "customer_id", "product_id", "product_name", "category",
               "quantity", "price", "revenue", "region", "status", "channel", "device",
               "created_at", "updated_at", "order_date", "_source_file", "_ingested_at"]

valid, rejected = validate(enrich(clean(bronze)))
silver_df = deduplicate(valid).select(SILVER_COLS)

print(f"Bronze    : {bronze.count():>6,} linhas")
print(f"Válidas   : {valid.count():>6,} linhas")
print(f"Rejeitadas: {rejected.count():>6,} linhas")
print(f"Silver    : {silver_df.count():>6,} pedidos únicos")

---
## 7. Os limites do Parquet puro

O pipeline funciona, mas tem problemas sérios em produção:

| Problema | Exemplo |
|----------|---------|
| **Sem atomicidade** | O job falha no meio do `overwrite`: a tabela fica pela metade |
| **Sem UPDATE/MERGE** | Uma atualização de status obriga a reescrever a partição (ou a tabela) |
| **Leituras inconsistentes** | Um leitor vê arquivos antigos e novos misturados durante a escrita |
| **Sem histórico** | Não dá para ver como a tabela estava ontem |
| **Schema sem proteção** | Um arquivo com colunas diferentes corrompe a tabela silenciosamente |

O **Delta Lake** resolve isso adicionando um **log de transações** ao lado dos arquivos Parquet.

```
  silver_delta/
  ├── _delta_log/
  │   ├── 00000000000000000000.json   <- versão 0: "adicionar arquivos A, B, C"
  │   ├── 00000000000000000001.json   <- versão 1: "remover B, adicionar D"  (MERGE)
  │   └── 00000000000000000002.json   <- versão 2: "remover A"               (DELETE)
  ├── order_date=2024-07-01/part-....parquet
  └── order_date=2024-07-02/part-....parquet
```

Os dados continuam em Parquet. O log diz **quais arquivos formam cada versão** da tabela. Escrever é atômico: ou o arquivo de log da nova versão existe, ou a versão não aconteceu.

---
## 8. Delta Lake na prática

### 8.1 Criar a tabela Silver em Delta

In [ ]:
SILVER_DELTA = f"{LAKE}/silver_delta/orders"

(silver_df
 .repartition("order_date")
 .write.format("delta")
 .mode("overwrite")
 .partitionBy("order_date")
 .save(SILVER_DELTA))

orders_dt = DeltaTable.forPath(spark, SILVER_DELTA)
print("Linhas:", orders_dt.toDF().count())
print("Arquivos de log:", sorted(os.listdir(f"{SILVER_DELTA}/_delta_log")))

In [ ]:
# O que está dentro de um commit
with open(f"{SILVER_DELTA}/_delta_log/00000000000000000000.json") as f:
    for line in f.readlines()[:4]:
        action = json.loads(line)
        print(list(action.keys())[0], "->", str(action)[:150])

### 8.2 Chegou um novo dia: MERGE (upsert)

No dia 04/07 chegam pedidos novos e **atualizações de pedidos antigos**. Com Parquet, teríamos que reprocessar tudo. Com Delta usamos `MERGE`:

```sql
MERGE INTO silver AS t
USING novos AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *   -- só se for mais recente
WHEN NOT MATCHED THEN INSERT *
```

A condição `s.updated_at > t.updated_at` protege contra dados atrasados: um evento antigo que chega depois não sobrescreve um estado mais novo.

In [ ]:
n_csv, n_json = generate_day(datetime.fromisoformat("2024-07-04"))
print(f"Novo dia: {n_csv} linhas CSV, {n_json} linhas JSON")

# Bronze incremental: lê só os arquivos novos e faz append
def read_raw(csv_glob, json_glob):
    c = (spark.read.option("header", True).csv(csv_glob)
         .select("*", F.col("_metadata.file_path").alias("_source_file")).withColumn("channel", F.lit("store")))
    j = (spark.read.option("primitivesAsString", True).json(json_glob)
         .select("*", F.col("_metadata.file_path").alias("_source_file")).withColumn("channel", F.lit("app")))
    return c.unionByName(j, allowMissingColumns=True).withColumn("_ingested_at", F.current_timestamp())

new_bronze = read_raw(f"{RAW}/orders_csv/pdv_20240704.csv", f"{RAW}/orders_json/app_20240704.json")
new_bronze.write.mode("append").parquet(BRONZE)

# Silver: mesmas funções de antes
new_valid, new_rejected = validate(enrich(clean(new_bronze)))
updates = deduplicate(new_valid).select(SILVER_COLS)
new_rejected.write.mode("append").parquet(QUARANTINE)

print("Registros para o MERGE:", updates.count())

In [ ]:
# collect() agora: o DataFrame do Delta sempre lê a versão mais recente quando é executado
status_before = spark.createDataFrame(orders_dt.toDF().groupBy("status").count().collect())

(orders_dt.alias("t")
 .merge(updates.alias("s"), "t.order_id = s.order_id")
 .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
 .whenNotMatchedInsertAll()
 .execute())

status_after = orders_dt.toDF().groupBy("status").count()
(status_before.withColumnRenamed("count", "antes")
 .join(status_after.withColumnRenamed("count", "depois"), "status", "full")
 .orderBy("status").show())

# Métricas do MERGE ficam registradas no histórico
(orders_dt.history(1)
 .select("version", "operation",
         F.col("operationMetrics")["numTargetRowsInserted"].alias("inseridas"),
         F.col("operationMetrics")["numTargetRowsUpdated"].alias("atualizadas"))
 .show())

### 8.3 Histórico e time travel

Cada operação gera uma nova versão. Versões antigas continuam legíveis até serem removidas pelo `VACUUM`.

In [ ]:
orders_dt.history().select("version", "timestamp", "operation").show(truncate=False)

v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_DELTA)
v1 = spark.read.format("delta").load(SILVER_DELTA)
print(f"Versão 0: {v0.count():,} pedidos | Versão atual: {v1.count():,} pedidos")

# Um pedido que mudou de status entre as versões
changed = (v0.alias("a").join(v1.alias("b"), "order_id")
           .filter(F.col("a.status") != F.col("b.status"))
           .select("order_id", F.col("a.status").alias("status_v0"), F.col("b.status").alias("status_atual")))
changed.show(5)

In [ ]:
# O mesmo em SQL
spark.sql(f"SELECT status, COUNT(*) AS n FROM delta.`{SILVER_DELTA}` VERSION AS OF 0 GROUP BY status").show()
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_DELTA}`").select("version", "operation").show()

### 8.4 Schema enforcement e schema evolution

O Delta **recusa** escritas com schema incompatível. Isso evita que um arquivo diferente corrompa a tabela. Quando a mudança é intencional, a evolução é explícita com `mergeSchema`.

In [ ]:
with_coupon = updates.limit(5).withColumn("coupon", F.lit("BLACKFRIDAY"))

try:
    with_coupon.write.format("delta").mode("append").save(SILVER_DELTA)
except Exception as e:
    print("Escrita recusada:", type(e).__name__)
    print(str(e).splitlines()[0][:200])

In [ ]:
# Evolução intencional: a coluna nova é adicionada e as linhas antigas recebem nulo
(with_coupon.write.format("delta")
 .mode("append")
 .option("mergeSchema", "true")
 .save(SILVER_DELTA))

spark.read.format("delta").load(SILVER_DELTA).groupBy("coupon").count().show()

> Esse append de 5 linhas duplicou pedidos de propósito para a demonstração. Vamos desfazê-lo na próxima seção.

### 8.5 DELETE, UPDATE e RESTORE

Exemplo típico: um cliente pede a exclusão dos seus dados (LGPD).

In [ ]:
# Desfaz o append da demonstração anterior voltando à versão do MERGE
merge_version = orders_dt.history().filter("operation = 'MERGE'").agg(F.max("version")).first()[0]
orders_dt.restoreToVersion(merge_version)
print("Restaurado para a versão", merge_version, "| linhas:", orders_dt.toDF().count())

# LGPD: apagar todos os pedidos de um cliente
customer = orders_dt.toDF().select("customer_id").first()[0]
print(f"Pedidos do cliente {customer} antes:", orders_dt.toDF().filter(F.col("customer_id") == customer).count())
orders_dt.delete(F.col("customer_id") == customer)
print(f"Pedidos do cliente {customer} depois:", orders_dt.toDF().filter(F.col("customer_id") == customer).count())

# UPDATE: pedidos 'created' há mais de 2 dias passam a 'expired'
orders_dt.update(
    condition="status = 'created' AND created_at < '2024-07-03'",
    set={"status": F.lit("expired")},
)
orders_dt.toDF().groupBy("status").count().orderBy("status").show()

**Atenção:** depois do `DELETE`, os dados do cliente ainda existem nos arquivos Parquet das versões antigas, e o time travel consegue lê-los. Para a exclusão ser real, é preciso rodar `VACUUM`.

### 8.6 OPTIMIZE e VACUUM

| Comando | O que faz | Quando |
|---------|-----------|--------|
| `OPTIMIZE` | Compacta arquivos pequenos em arquivos maiores | Após muitos MERGEs/appends |
| `OPTIMIZE ... ZORDER BY (col)` | Também reorganiza os dados para pular mais blocos em filtros por `col` | Colunas muito filtradas, fora da partição |
| `VACUUM` | Apaga do disco os arquivos que nenhuma versão recente usa | Periodicamente; padrão mantém 7 dias |

In [ ]:
print("Arquivos antes do OPTIMIZE:", orders_dt.detail().first()["numFiles"])
orders_dt.optimize().executeZOrderBy("region")
print("Arquivos depois do OPTIMIZE:", orders_dt.detail().first()["numFiles"])

def parquet_files(path):
    return sum(f.endswith(".parquet") for _, _, fs in os.walk(path) for f in fs)

print("\nArquivos .parquet no disco antes do VACUUM:", parquet_files(SILVER_DELTA))

# Só para a aula: retenção zero. Em produção, mantenha o padrão (168 horas).
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
orders_dt.vacuum(0)
print("Arquivos .parquet no disco depois do VACUUM:", parquet_files(SILVER_DELTA))

try:
    n = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_DELTA).count()
    print("\nVersão 0 ainda legível:", n)
except Exception as e:
    print("\nTime travel para a versão 0 não é mais possível: os arquivos foram apagados.")

### 8.7 Gold a partir da Silver Delta

In [ ]:
GOLD_DELTA = f"{LAKE}/gold_delta/daily_sales"

(spark.read.format("delta").load(SILVER_DELTA)
 .filter(~F.col("status").isin("cancelled", "expired"))
 .groupBy("order_date", "region")
 .agg(F.count("*").alias("orders"), F.round(F.sum("revenue"), 2).alias("revenue"))
 .write.format("delta").mode("overwrite").save(GOLD_DELTA))

spark.read.format("delta").load(GOLD_DELTA).orderBy("order_date", "region").show(10)

---
## 9. Exercícios

In [ ]:
# ── Exercício 4 (Delta) ─────────────────────────────────────────────────────
# Simule um evento ATRASADO: pegue um pedido já "shipped" ou "delivered" e crie uma
# versão dele com status "paid" e updated_at igual ao created_at (ou seja, mais antiga).
# Faça o MERGE e confirme que ele NÃO sobrescreveu o status atual.
# Use o history() para mostrar quantas linhas foram atualizadas.

# Sua solução:

### Soluções

In [ ]:
# ── Solução 4 ───────────────────────────────────────────────────────────────
current = orders_dt.toDF().filter(F.col("status").isin("shipped", "delivered")).first()
print("Estado atual:", current["order_id"], current["status"], current["updated_at"])

late_event = (
    orders_dt.toDF().filter(F.col("order_id") == current["order_id"])
    .withColumn("status", F.lit("paid"))
    .withColumn("updated_at", F.col("created_at"))      # mais antigo que o estado atual
)

(orders_dt.alias("t")
 .merge(late_event.alias("s"), "t.order_id = s.order_id")
 .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
 .whenNotMatchedInsertAll()
 .execute())

print("Depois do MERGE:", orders_dt.toDF().filter(F.col("order_id") == current["order_id"]).first()["status"])
orders_dt.history(1).select("operation", F.col("operationMetrics")["numTargetRowsUpdated"].alias("atualizadas")).show()

---
## Resumo

| Camada | Formato | Conteúdo | Operações típicas |
|--------|---------|----------|-------------------|
| Bronze | Parquet/Delta, append | Dado bruto, tudo string, + linhagem | `read`, `unionByName`, `_metadata` |
| Silver | Delta, particionada | Tipado, validado, uma linha por entidade | `try_cast`, regras, `row_number`, `MERGE` |
| Quarentena | Parquet/Delta | Rejeitados + motivo | `array_compact`, `explode` |
| Gold | Delta | Métricas para o negócio | `groupBy`, window functions |

| Delta Lake | Comando |
|------------|---------|
| Upsert | `DeltaTable.merge(...).whenMatchedUpdateAll().whenNotMatchedInsertAll()` |
| Histórico | `.history()` / `DESCRIBE HISTORY` |
| Time travel | `.option("versionAsOf", n)` / `VERSION AS OF n` |
| Desfazer | `.restoreToVersion(n)` |
| Evoluir schema | `.option("mergeSchema", "true")` |
| Compactar | `.optimize().executeCompaction()` / `.executeZOrderBy(col)` |
| Limpar versões antigas | `.vacuum(horas)` |

**Próxima aula:** Structured Streaming, o mesmo pipeline processando dados que chegam continuamente.

In [ ]:
spark.stop()